# Observation-level schema validation dataframe

Create one dataframe row per validation observation. Observation and candidate columns retain the exact Structured Output/Pydantic field names and values; candidate columns are empty when an observation has no candidate. `review_family` is a manually defined analysis aid derived from `proposed_name`, not a model output.

In [1]:
import json
from pathlib import Path
from typing import Any

import pandas as pd

from data_snapshot.constants import ROOT


pd.set_option('display.max_colwidth', None)

In [2]:
RESULTS_PATH = ROOT / "notebooks/schema_validation/outputs/results.jsonl"


In [3]:
CANDIDATE_FAMILIES: dict[str, set[str]] = {
    "Source-document retrieval": {
        "source_document_url",
        "source_document_locator",
    },
    "Source-document identity and type": {
        "source_document_identifier",
        "source_document_type",
    },
    "Source-document publication date": {
        "source_document_publication_date",
        "source_document_date",
    },
    "Source-document attribution": {
        "source_document_creator",
        "source_document_contributor",
        "source_document_author",
        "source_document_producer",
        "source_document_organization",
        "source_document_attribution",
    },
    "Snapshot artifact date": {
        "snapshot_date",
        "snapshot_production_date",
        "snapshot_creation_date",
    },
    "Snapshot artifact attribution": {
        "snapshot_creator",
        "snapshot_producer",
        "snapshot_contributor",
        "snapshot_attribution",
    },
    "Cartographic scale and coordinates": {
        "map_scale",
        "cartographic_scale",
        "map_scale_unit",
        "coordinate_grid",
        "coordinate_extent",
    },
    "Analytical variable role": {
        "axis_variable",
        "dependent_variable",
        "explanatory_variable",
        "outcome_variable",
        "model_variable_role",
    },
    "Classification semantics": {
        "classification_system",
        "classification_level",
        "classification_rule",
        "category_hierarchy",
    },
    "Composite visualization structure": {
        "visualization_composition",
        "visualization_component_type",
        "visualization_components",
        "embedded_visualization_type",
    },
    "Data-collection timing and responsibility": {
        "data_collection_frequency",
        "data_collection_schedule",
        "data_collection_responsibility",
        "data_collection_responsible_party",
    },
    "Sample size": {"sample_size"},
    "Uncertainty representation": {"uncertainty_representation"},
    "Reference event or context": {
        "reference_event",
        "temporal_reference_event",
        "event_context",
    },
}

FAMILY_BY_CANDIDATE = {
    proposed_name: family
    for family, proposed_names in CANDIDATE_FAMILIES.items()
    for proposed_name in proposed_names
}


In [4]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    """Load JSON objects from a JSONL file.

    Parameters
    ----------
    path : Path
        JSONL file to read.

    Returns
    -------
    list[dict[str, Any]]
        Parsed records.

    Raises
    ------
    FileNotFoundError
        If the JSONL file does not exist.
    ValueError
        If a non-empty line is not valid JSON.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Results file not found: {path}")

    records = []
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path} at line {line_number}."
                ) from exc
    return records


def build_observation_dataframe(results: list[dict[str, Any]]) -> pd.DataFrame:
    """Build one dataframe row per validation observation.

    Parameters
    ----------
    results : list[dict[str, Any]]
        Successful parsed validation records.

    Returns
    -------
    pandas.DataFrame
        Observation-level table using exact structured-output field names.

    Raises
    ------
    ValueError
        If a candidate references an unknown observation, an observation supports
        multiple candidates, or an observation key is duplicated.
    """
    rows = []
    for record in results:
        parsed = record["parsed_output"]
        observations = parsed["observations"]
        observation_ids = {item["observation_id"] for item in observations}
        candidate_by_observation = {}

        for candidate in parsed["candidate_new_fields"]:
            for observation_id in candidate["supporting_observation_ids"]:
                if observation_id not in observation_ids:
                    raise ValueError(
                        f"Unknown observation {observation_id!r} in "
                        f"{record['snapshot_file_name']}."
                    )
                if observation_id in candidate_by_observation:
                    raise ValueError(
                        f"Observation {observation_id!r} supports multiple "
                        f"candidates in {record['snapshot_file_name']}."
                    )
                candidate_by_observation[observation_id] = candidate

        for observation in observations:
            candidate = candidate_by_observation.get(observation["observation_id"], {})
            rows.append(
                {
                    "snapshot_file_name": record["snapshot_file_name"],
                    "source": record["source"],
                    "artifact_type": record["artifact_type"],
                    **observation,
                    "proposed_name": candidate.get("proposed_name"),
                    "review_family": FAMILY_BY_CANDIDATE.get(
                        candidate.get("proposed_name")
                    ),
                    "definition": candidate.get("definition"),
                    "supporting_observation_ids": candidate.get(
                        "supporting_observation_ids"
                    ),
                    "why_existing_fields_are_insufficient": candidate.get(
                        "why_existing_fields_are_insufficient"
                    ),
                    "operational_value": candidate.get("operational_value"),
                    "source_level": candidate.get("source_level"),
                }
            )

    dataframe = (
        pd.DataFrame(rows)
        .sort_values(
            ["source", "artifact_type", "snapshot_file_name", "observation_id"]
        )
        .reset_index(drop=True)
    )
    key_columns = ["source", "snapshot_file_name", "observation_id"]
    if dataframe.duplicated(key_columns).any():
        raise ValueError("Observation keys must be unique.")
    return dataframe


In [5]:
results = load_jsonl(RESULTS_PATH)
df = build_observation_dataframe(results)
df


,snapshot_file_name,source,artifact_type,observation_id,metadata_concept,evidence,evidence_source,closest_schema_fields,fit_status,fit_rationale,proposed_name,review_family,definition,supporting_observation_ids,why_existing_fields_are_insufficient,operational_value,source_level
0,document_10240515_figure_000.png,prwp,figure,O1,Snapshot title and document-assigned figure nu...,The snapshot heading reads “Figure 1 – Tourism...,snapshot,"[title, internal_identifier]",covered,The heading provides both the primary identify...,None,None,None,None,None,None,None
1,document_10240515_figure_000.png,prwp,figure,O10,Interpretive footnote for exceptional plotted ...,The snapshot includes the note “(*) Actual rat...,snapshot,[interpretive_note],covered,The existing interpretive_note field captures ...,None,None,None,None,None,None,None
2,document_10240515_figure_000.png,prwp,figure,O11,Parent source document title,The source-document metadata title is “Foreign...,document,[source_document_title],covered,The existing source_document_title field direc...,None,None,None,None,None,None,None
3,document_10240515_figure_000.png,prwp,figure,O12,Snapshot language,The snapshot text is in English and the source...,both,[language],covered,The existing language field captures the langu...,None,None,None,None,None,None,None
4,document_10240515_figure_000.png,prwp,figure,O13,Parent source document identifier,The source-document metadata lists identifiers...,document,"[source_document_title, internal_identifier, p...",no_fit,Existing identifier fields refer to the snapsh...,source_document_identifier,Source-document identity and type,A formal identifier assigned to the parent sou...,[O13],internal_identifier is defined for identifiers...,"Supports reliable linking, deduplication, cita...",document
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2871,zone_frontalire_nord_kivu_-_ituri_rpublique_dm...,unhcr,table,obs_08,Language used in the snapshot,"The table text is in French, with phrases such...",both,[language],covered,The existing language field directly captures ...,None,None,None,None,None,None,None
2872,zone_frontalire_nord_kivu_-_ituri_rpublique_dm...,unhcr,table,obs_09,Parent source document title,The source-document metadata title is “Zone Fr...,document,[source_document_title],covered,The existing source_document_title field direc...,None,None,None,None,None,None,None
2873,zone_frontalire_nord_kivu_-_ituri_rpublique_dm...,unhcr,table,obs_10,Retrieval URL for the parent source document,The document metadata provides a ReliefWeb URL...,document,[source_document_title],no_fit,The schema has a field for the source document...,source_document_url,Source-document retrieval,A URL where the parent source document or its ...,[obs_10],source_document_title identifies the parent do...,"Supports retrieval, auditability, citation, an...",document
2874,zone_frontalire_nord_kivu_-_ituri_rpublique_dm...,unhcr,table,obs_11,Publication date of the parent source document,The source-document metadata lists date_publis...,document,[],no_fit,The schema has temporal fields for represented...,source_document_publication_date,Source-document publication date,The publication or posting date of the parent ...,[obs_11],time_period describes the period represented b...,Helps distinguish data-period metadata from so...,document


# Review by candidate families

In [9]:
list(CANDIDATE_FAMILIES.keys())

['Source-document retrieval',
 'Source-document identity and type',
 'Source-document publication date',
 'Source-document attribution',
 'Snapshot artifact date',
 'Snapshot artifact attribution',
 'Cartographic scale and coordinates',
 'Analytical variable role',
 'Classification semantics',
 'Composite visualization structure',
 'Data-collection timing and responsibility',
 'Sample size',
 'Uncertainty representation',
 'Reference event or context']

In [28]:
x = "Classification semantics"
display(CANDIDATE_FAMILIES[x])
display(df[df["review_family"] == x])

{'category_hierarchy',
 'classification_level',
 'classification_rule',
 'classification_system'}

,snapshot_file_name,source,artifact_type,observation_id,metadata_concept,evidence,evidence_source,closest_schema_fields,fit_status,fit_rationale,proposed_name,review_family,definition,supporting_observation_ids,why_existing_fields_are_insufficient,operational_value,source_level
81,document_11225119_figure_000.png,prwp,figure,O6,Classification or coding system for categories,The category axis and table explicitly use “hscodes” with eight-digit product codes.,snapshot,"[category_dimension, category_labels]",weak_fit,Existing category fields can store the codes as labels but do not separately represent that the categories belong to a named classification system.,classification_system,Classification semantics,"The named taxonomy, coding scheme, or classification system used to define categorical labels represented in the snapshot.",[O6],"category_dimension and category_labels can store the dimension and labels, but they do not separately encode that labels come from a reusable code system such as HS codes.","Supports retrieval, harmonization, and interpretation of snapshots that use standardized classifications such as product, industry, occupation, disease, or geographic codes.",snapshot
165,document_11867889_figure_016.png,prwp,figure,obs_011,Commodity classification standard and version used for the represented categories,The source note explicitly includes “(SITC version 2).”,snapshot,"[interpretive_note, data_source, category_dimension]",weak_fit,"Existing fields can only store the classification standard as an unstructured note or conflate it with the data source or category dimension, losing the distinct coding-system role.",classification_system,Classification semantics,"The named classification, taxonomy, coding scheme, or version used to define or group categories represented in the snapshot.",[obs_011],"interpretive_note can preserve the text only as an unstructured caveat, data_source would conflate a classification scheme with data provenance, and category_dimension does not identify the external coding standard or version.","This field would support retrieval, comparison, and reuse of snapshots whose categories depend on specific classification systems or versions, such as trade, industry, occupation, disease, or education taxonomies.",snapshot
269,document_15151160_figure_004.png,prwp,figure,obs_06,Non-geographic classification system and reporting level,"The caption specifies that the product lines are reported at the ""2 digit HS level.""",snapshot,"[category_dimension, category_labels]",weak_fit,Category fields can mention HS codes but do not cleanly capture the separate taxonomy and classification level used for the categories.,classification_level,Classification semantics,"The named non-geographic categorical classification system and reporting level used to classify represented entities, such as HS 2 digit product codes.",[obs_06],"category_dimension and category_labels can describe the organizing dimension and visible labels, but they do not separately preserve the taxonomy and level of aggregation that define how the categories should be interpreted.","Supports retrieval, comparison, and reuse of snapshots that use standardized product, industry, occupation, or other coded classification systems at different levels of detail.",snapshot
433,document_7581269_figure_002.png,prwp,figure,O3,Explicit threshold rule defining highlighted map areas,"The subtitle specifies the displayed areas as ""Districts reporting more than 100 deaths since 1996.""",snapshot,"[variable_name, category_labels, interpretive_note]",weak_fit,"Existing fields can mention the threshold only by embedding it in a variable, label, or note, but none directly captures a classification or inclusion rule used to define displayed entities.",classification_rule,Classification semantics,"An explicit rule, threshold, or condition used to classify, filter, or determine which entities or observations are displayed or highlighted in the snapshot.",[O3]

# Review by unfamilied

In [30]:
df.loc[df["proposed_name"].notna() & df["review_family"].isna()].sort_values(
    ["source", "snapshot_file_name"]
)

,snapshot_file_name,source,artifact_type,observation_id,metadata_concept,evidence,evidence_source,closest_schema_fields,fit_status,fit_rationale,proposed_name,review_family,definition,supporting_observation_ids,why_existing_fields_are_insufficient,operational_value,source_level
474,document_10087951_table_006.png,prwp,table,obs_008,Regression model terms and covariates represented as rows,"Rows include terms such as “Second round * actual intervention,” “Full exposure * actual intervention,” “Male Child,” “Twin,” and education variables.",snapshot,"[row_dimension, category_labels, variable_name]",weak_fit,"Row_dimension can say rows are model terms, but existing fields do not cleanly represent included covariates or interaction terms without conflating them with categories or the primary outcome variable.",model_term,None,"An explanatory variable, interaction term, constant, or other analytical term explicitly represented in a model-results snapshot.",[obs_008],"Row_dimension can describe the row concept at a high level, category_labels are defined as category names for a category dimension, and variable_name is reserved for the primary represented variable, so none adequately captures analytical model terms without semantic distortion.","Supports indexing and reuse of regression or model-results snapshots by included covariates, treatment variables, and interaction terms while distinguishing them from the primary outcome.",snapshot
13,document_10240515_figure_000.png,prwp,figure,O9,Data derivation or calculation statement,The source line states “Authors’ calculations using Caribbean Tourism Organization data.”,snapshot,"[data_source, analysis_method, interpretive_note]",weak_fit,"Existing fields can capture the named source or a specific method, but not the provenance fact that the displayed values are author-calculated from source data.",data_derivation_statement,None,"An explicit statement describing that represented values were derived, calculated, transformed, or otherwise produced from cited source data, including the responsible party when stated.",[O9],"data_source captures the underlying cited data, analysis_method captures named analytical methods, and interpretive_note captures explanatory caveats, but none cleanly represents calculation provenance such as “Authors’ calculations using … data.”","Helps users distinguish directly reported source data from values computed or transformed by document authors, supporting interpretation, verification, and reuse.",snapshot
49,document_10455617_figure_000.png,prwp,figure,O5,Analytical overlay or fitted trend line,A red downward-sloping line is drawn across the scatter plot in addition to the country points.,snapshot,"[visualization_type, analysis_method]",weak_fit,"visualization_type can note the scatter plot and analysis_method applies only when a method is stated, but neither cleanly captures the explicit presence of an analytical trend-line overlay.",analytical_overlay,None,"An explicitly visible analytical layer added to a visualization, such as a fitted trend line, reference line, confidence band, or smoothing curve.",[O5],"visualization_type captures the primary chart form and analysis_method requires a stated method, but neither records the presence of an interpretive overlay when the method is not specified.",Supports discovery and interpretation of charts that present modeled or summarized relationships in addition to raw plotted values.,snapshot
53,document_10455617_figure_000.png,prwp,figure,O9,Sectoral coverage of the represented services data,"The note states that the covered sectors are ""financial, telecommunications, retailing, maritime, air passenger transport, and professional services.""",snapshot,"[subject_domain, subject_summary, category_labels]",weak_fit,"subject fields can indicate a services-trade topic, but the schema lacks a structured field for the specific sectors covered when they are not visible categories in the chart.",sector_coverage,None,"T

# Review by boundary-quality rows

In [32]:
df.loc[df["fit_status"].isin(["uncertain", "out_of_scope"])].sort_values(
    ["source", "snapshot_file_name"]
)

,snapshot_file_name,source,artifact_type,observation_id,metadata_concept,evidence,evidence_source,closest_schema_fields,fit_status,fit_rationale,proposed_name,review_family,definition,supporting_observation_ids,why_existing_fields_are_insufficient,operational_value,source_level
31,document_10395051_figure_007.png,prwp,figure,O12,Document-level analytical method applicability,"The document abstract states ""Using cross-country regression analysis,"" but the snapshot itself does not state a method.",document,[analysis_method],uncertain,The evidence identifies a method used in the paper but is insufficient to reliably assign that method to this specific snapshot.,None,None,None,None,None,None,None
522,document_11017288_table_007.png,prwp,table,obs_15,Temporal period associated with the parent study,"The source document title includes ""2001-2008,"" while the table itself does not state a data period.",document,[time_period],uncertain,The evidence establishes a parent-document study period but is insufficient to reliably assign it as the snapshot's represented data period.,None,None,None,None,None,None,None
58,document_11174028_figure_004.png,prwp,figure,obs_005,Subnational geographic reporting granularity,"The maps show many internal administrative-looking polygons within Indonesia, but no label identifies the boundary level.",snapshot,[geographic_granularity],uncertain,"The existing geographic_granularity field would cover the concept if the administrative level were stated, but the evidence does not identify the level reliably.",None,None,None,None,None,None,None
60,document_11174028_figure_004.png,prwp,figure,obs_007,Unidentified mapped variable or indicator,"The snapshot shows shaded areas but includes no visible legend, caption, or variable label identifying what the shading measures.",snapshot,"[variable_name, unit_of_measure, measure_type]",uncertain,"The schema has fields for the represented variable and measurement context, but the evidence is insufficient to determine those metadata values.",None,None,None,None,None,None,None
553,document_11174028_table_006.png,prwp,table,obs_16,Regression or model output statistics,"The table includes labels such as ""Coefficients,"" ""Coef,"" ""SE,"" ""Observations,"" ""Households,"" and ""R² (within).""",snapshot,[],out_of_scope,These are statistical outputs or model-result contents rather than metadata fields to be represented by the schema.,None,None,None,None,None,None,None
565,document_11174028_table_009.png,prwp,table,obs_012,statistical estimates and model diagnostics as table contents,"The table displays “Coef,” “SE,” “Observations,” “Households,” and “R² (within)” with associated numeric values.",snapshot,[],out_of_scope,"The displayed estimates, standard errors, counts, and diagnostics are statistical outputs or table contents explicitly outside the metadata schema scope.",None,None,None,None,None,None,None
76,document_11225119_figure_000.png,prwp,figure,O12,Analytical tool or simulation context,The source-document title says the paper assesses trade policy changes “using TRIST (tariff reform impact simulation tool).”,document,[analysis_method],uncertain,"The evidence identifies the document’s analytical tool, but it does not explicitly state that this specific figure was produced by TRIST.",None,None,None,None,None,None,None
785,document_443630_table_016.png,prwp,table,obs_15,Analytical method identification,"The table contains ""Coef.,"" ""Dependent Variable,"" ""R²,"" and ""Mill's Ratio,"" but no method name is stated in the snapshot.",snapshot,[analysis_method],uncertain,The evidence suggests model-based analysis but is insufficient to reliably identify an explicit analytical method without inference.,None,None,None,None,None,None,None
786,document_443630_table_016.png,prwp,table,obs_16,Possible data-period cue in cited source name,"The source note says the calculations are based on ""ENIGH 96 and DGPyP, SEP.""",snapshot,"[time_period, data_source]",uncertain,"The string